### Import Libraries

In [2]:
import os
import numpy as np
import pandas as pd
import seaborn 
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

### Import data

In [3]:
image_paths = []
labels = []
classes = 43
base_path = Path("data_gtsrb/Train")

for i in range(classes):
    path = base_path / str(i)
    images = os.listdir(path) 

    for j in images:
        try:
            image_file_path = path / j
            image_paths.append(str(image_file_path))
            labels.append(i)
        except:
            pass


train_paths, test_paths, y_train, y_test = train_test_split(
    image_paths, labels, test_size=0.2, random_state=42, stratify=labels)

### Dataset and dataloader

In [4]:
class GTSRBDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(), 
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

In [12]:
train_dataset = GTSRBDataset(train_paths, y_train, transform=data_transforms)
test_dataset = GTSRBDataset(test_paths, y_test, transform=data_transforms)

batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

### Model

In [13]:
class MobileNetV2TrafficSigns(nn.Module):
    def __init__(self, num_classes=43):
        super(MobileNetV2TrafficSigns, self).__init__()

        weights = MobileNet_V2_Weights.DEFAULT
        self.base_model = mobilenet_v2(weights=weights)

        for param in self.base_model.parameters():
            param.requires_grad = False # freeze weights

        # allow to change only the final layer
        in_features = self.base_model.classifier[1].in_features
        self.base_model.classifier[1] = nn.Linear(in_features, num_classes)

    def forward(self, x):
        x = self.base_model(x)
        return x

In [14]:
model = MobileNetV2TrafficSigns(num_classes=43)

print(model.base_model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=43, bias=True)
)


### Train model

In [15]:
from tqdm import tqdm
import sys

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

epoch = 5
print(f"Training is run on: {device}")

for i in range(epoch):
    model.train()
    running_loss = 0.0
    
    loop = tqdm(train_loader, desc=f"Epoch [{i+1}/{epoch}]", leave=True, file=sys.stdout)
    
    for image, label in loop:
        image = image.to(device)
        label = label.to(device)
        
        optimizer.zero_grad()
        pred = model(image)
        loss = criterion(pred, label)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        loop.set_postfix(loss=loss.item())
        
    print(f'Epoch [{i+1}/{epoch}] | Training loss: {running_loss/len(train_loader):.4f}')

Training is run on: cpu
Epoch [1/5]: 100%|██████████| 123/123 [13:01<00:00,  6.35s/it, loss=1.49]
Epoch [1/5] | Training loss: 2.1829
Epoch [2/5]:  20%|██        | 25/123 [02:57<12:03,  7.38s/it, loss=1.27]

In [ ]:
model.eval()

In [ ]:
torch.save(model.state_dict(), "our_model.pth")